# 01. Introducción a `PINN_V4`, Geometría 2D y Formato JSON

En este cuaderno aprenderá a:
1. Explorar los casos curados incluidos en `cases/benchmarks/` y `cases/thesis_curated/`.
2. Diseñar y modificar casos 2D mediante `CaseEditorHelper` usando **constantes**, **expresiones `(x, y)`**, **lógica condicional `piecewise`** y **tablas 1D/2D**.
3. Inspeccionar la geometría y heterogeneidad con el visor interactivo Plotly (zoom + anotaciones).
4. Guardar el problema en `frontend/workspace/user_cases/` para utilizarlo desde Python, CLI o Streamlit.

In [1]:
import sys
from pathlib import Path

# Localiza PINN_V4 automáticamente aunque se ejecute desde dentro de notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("Raíz PINN_V4:", ROOT)

Raíz PINN_V4: C:\usr\ths_mia_fiis\tesis_MIA_lk\tesis_MIA_2026-2_pinn


In [2]:
from pinn_thermal.core.device import DeviceManager
from pinn_thermal.core.problem import CaseCatalog
from pinn_thermal.visualization import CaseEditorHelper, InteractivePlotter
from frontend.state import WorkspaceStore

print("Hardware PyTorch:", DeviceManager.describe())
catalog = CaseCatalog(ROOT / "cases")
entries = catalog.list_entries()
print(f"Total de casos en catálogo: {len(entries)}")
for e in entries[:8]:
    print(f" - [{e['category']}] {e['id']}: {e['title']}")

Hardware PyTorch: {'torch_version': '2.9.0+xpu', 'backends': {'cuda': {'available': False, 'count': 0, 'kind': 'nvidia_cuda', 'devices': []}, 'xpu': {'available': True, 'count': 1, 'devices': ['Intel(R) Graphics']}, 'npu': {'available': False, 'count': 0, 'devices': []}, 'mps': {'available': False, 'count': 0, 'devices': []}, 'dml': {'available': False, 'count': 0, 'devices': []}, 'cpu': {'available': True, 'count': 14, 'devices': ['CPU']}}, 'recommended_device': 'xpu', 'cpu_threads': 12}
Total de casos en catálogo: 29
 - [benchmarks] annulus: annulus
 - [benchmarks] cable_disk: Single circular conductor in homogeneous soil
 - [benchmarks] mms_constant: mms_constant
 - [benchmarks] mms_high_contrast: mms_high_contrast
 - [benchmarks] mms_interface: mms_interface
 - [benchmarks] mms_layered_y: mms_layered_y
 - [benchmarks] mms_robin: mms_robin
 - [benchmarks] mms_sine: Manufactured sine field on a unit square


## Diseño guiado de un caso con suelo heterogéneo (tabla 2D + `piecewise`) y anotaciones

In [3]:
helper = CaseEditorHelper(CaseEditorHelper.template_buried_single_cable())
helper.set_metadata(
    case_id="demo_cuaderno_heterogeneo",
    title="Cable enterrado con relleno lógico y suelo por tabla 2D",
    family="cable",
)
# Suelo con tabla 2D bilineal
helper.upsert_region(
    region_id="soil",
    kind="soil",
    priority=0,
    shape={"type": "rectangle", "bounds": [-1.5, 1.5, -2.0, 0.0]},
    conductivity=helper.field_table_2d(
        xs=[-1.5, 0.0, 1.5],
        ys=[-2.0, -1.0, 0.0],
        matrix=[[0.75, 0.85, 0.75], [1.05, 1.25, 1.05], [1.4, 1.6, 1.4]],
    ),
    source=0.0,
)
# Relleno térmico con regla condicional lógica
helper.upsert_region(
    region_id="backfill",
    kind="soil",
    priority=1,
    shape={"type": "rectangle", "bounds": [-0.35, 0.35, -1.25, -0.75]},
    conductivity=helper.field_piecewise(
        [("y <= -1.0 and abs(x) < 0.25", 2.4)],
        default=1.8,
    ),
)
helper.add_annotation(0.0, -1.0, "Conductor XLPE (Q=45 kW/m³)")
assert helper.validate() == []

problem = helper.build_problem()
store = WorkspaceStore(ROOT / "frontend" / "workspace", ROOT / "cases")
saved_path = store.save_user_case(helper)
print("Caso guardado y disponible también en Streamlit:", saved_path)

Caso guardado y disponible también en Streamlit: C:\usr\ths_mia_fiis\tesis_MIA_lk\tesis_MIA_2026-2_pinn\frontend\workspace\user_cases\demo_cuaderno_heterogeneo.json


In [4]:
# Visor interactivo desacoplado (funciona en Jupyter y en Streamlit)
fig = InteractivePlotter.geometry_figure(problem, background="conductivity")
# En modo no interactivo / CI solo verificamos su estructura; en Jupyter renderiza con fig.show()
print("Figura Plotly generada con", len(fig.data), "trazas y", len(fig.layout.annotations), "anotaciones.")

Figura Plotly generada con 1 trazas y 2 anotaciones.


In [5]:
fig